# Raw to Silver: Data Preprocessing
This notebook demonstrates how to preprocess transaction data from the raw layer and save a cleaned version to the silver layer. You can adapt this workflow for other raw datasets in your project.

In [1]:


import pandas as pd

RAW_PATH = "../raw-data/synthetic_outerwear_sri_lanka_with_shop_ids.csv"
SILVER_PATH = "../Silver-data/products_clean.csv"

# Load raw products data
products = pd.read_csv(RAW_PATH)
print("Raw products Shape:", products.shape)
products.head()

Raw products Shape: (2625, 13)


,product_id,shop_id,name,category,color,fabric,size_range,price_LKR,style_tags,product_url,created_ts,popularity_score,stock_count
0,1.0,8.0,NaN,BEACH WEAR,Navy,Wool,L,3000.0,"Tall Friendly, Checked, Monochrome, Sporty, Da...",https://casualcorner.com/product/1,19:51.6,3.4,28.0
1,2.0,15.0,Luxury Rustic Coats,COATS,Checked,Polyester,XL-XXL,14000.0,"Floral, Formal, Comfy, Checked, Petite Friendl...",https://arena.com/product/2,46:58.1,4.6,15.0
2,3.0,3.0,Vintage Oversized Beach Wear,BEACH WEAR,Mustard,Velvet,XS-S,5000.0,"Dark Skin, Eco-friendly, Edgy, Hipster, Busine...",https://hameedia.com/product/3,NaN,2.7,39.0
3,4.0,12.0,Sporty Textured T-Shirts,T-SHIRTS,Brown,Silk,L,2500.0,"Rainy Weather, Festival, Fair Skin, Winter, Co...",NaN,32:39.4,4.9,5.0
4,5.0,21.0,Casual Classic Cardigans,CARDIGANS,Multi-color,Tencel,M,9000.0,"Neutral Skin, Business Casual, Formal, Winter,...",https://elements.com/product/5,31:05.6,2.9,18.0


## Step 1: Remove Duplicates

In [3]:
# Remove duplicate rows
products = products.drop_duplicates()
print("After removing duplicates:", products.shape)

After removing duplicates: (2559, 13)


## Step 2: Handle Missing Values

In [5]:
# Check for missing values
missing_summary = products.isnull().sum()
print("Missing values per column:\n", missing_summary)


Missing values per column:
 product_id          76
shop_id             83
name                73
category            77
color               69
fabric              74
size_range          75
price_LKR           74
style_tags          93
product_url         69
created_ts          77
popularity_score    90
stock_count         78
dtype: int64


In [6]:
# Drop rows where product_id is missing
products = products.dropna(subset=['product_id'])

# Fill textual columns
products['name'] = products['name'].fillna('UNKNOWN_PRODUCT')
products['category'] = products['category'].fillna('Miscellaneous')
products['color'] = products['color'].fillna('Unknown')
products['fabric'] = products['fabric'].fillna('Unknown')
products['size_range'] = products['size_range'].fillna('One Size')
products['style_tags'] = products['style_tags'].fillna('General')
products['product_url'] = products['product_url'].fillna('No URL')

# Fill numeric columns
products['price_LKR'] = products['price_LKR'].fillna(products['price_LKR'].median())
products['popularity_score'] = products['popularity_score'].fillna(0)
products['stock_count'] = products['stock_count'].fillna(0)

# Fill datetime
products['created_ts'] = pd.to_datetime(products['created_ts'], errors='coerce')
products['created_ts'] = products['created_ts'].fillna(pd.Timestamp('2022-01-01'))

# Fill shop_id (optional, depends on transactions)
products['shop_id'] = products['shop_id'].fillna(-1)  # or Unknown ID

# Verify
print(products.isna().sum())


product_id          0
shop_id             0
name                0
category            0
color               0
fabric              0
size_range          0
price_LKR           0
style_tags          0
product_url         0
created_ts          0
popularity_score    0
stock_count         0
dtype: int64


/var/folders/l_/c9wspj0n4453_nqxd1zd06d40000gn/T/ipykernel_62184/1501647860.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  products['created_ts'] = pd.to_datetime(products['created_ts'], errors='coerce')


## Step 3: Data Type Corrections (if needed)

In [12]:
products.dtypes

product_id                   int64
shop_id                      int64
name                        object
category                    object
color                       object
fabric                      object
size_range                  object
price_LKR                  float64
style_tags                  object
product_url                 object
created_ts          datetime64[ns]
popularity_score           float64
stock_count                  int64
dtype: object

In [9]:
products['product_id'] = products['product_id'].astype(int)
products['shop_id'] = products['shop_id'].astype(int)


In [11]:
products['stock_count'] = products['stock_count'].astype(int)


## Step 4: Save Cleaned Data to Silver Layer

In [13]:
# Save the cleaned dataframe to the silver layer
import os
os.makedirs(os.path.dirname(SILVER_PATH), exist_ok=True)
products.to_csv(SILVER_PATH, index=False)
print(f"Cleaned products data saved to {SILVER_PATH}")

Cleaned products data saved to ../Silver-data/products_clean.csv


---

Repeat this process for each raw dataset (users, products, shops, etc.) by changing the input/output paths and adapting the cleaning steps as needed.